In [1]:
import struct
import numpy as np
import pandas as pd
import scipy.stats as ss

class StandaloneSignalLossQuantifier:
    """
    A lightweight, standalone NDF parser designed specifically to extract
    signal loss metrics (corrupted messages and time gaps) without requiring
    the pyecog2 package.
    """
    
    def __init__(self, filepath, gap_threshold_seconds=1.0):
        self.filepath = filepath
        self.gap_threshold = gap_threshold_seconds
        
        # Firmware dependent clock variables from source
        self.clock_tick_cycle = 7.8125e-3  
        self.clock_division = self.clock_tick_cycle / 256.0

    def quantify(self):
        # 1. Read Metadata to find Data Address
        with open(self.filepath, 'rb') as f:
            f.seek(0)
            identifier = f.read(4)
            if identifier != b' ndf':
                raise ValueError("Not a valid NDF file.")
            
            f.read(4) # skip meta_data_string_address
            data_address = struct.unpack('>I', f.read(4))[0]

        # 2. Extract Raw Binary Data
        with open(self.filepath, 'rb') as f:
            f.seek(data_address)
            e_bit_reads = np.fromfile(f, dtype='u1')

        transmitter_id_bytes = e_bit_reads[::4]
        t_stamps_256 = e_bit_reads[3::4]

        # 3. Detect Frequencies and Valid Transmitters
        tid_message_counts = pd.Series(transmitter_id_bytes).value_counts()
        possible_freqs = [256, 512, 1024]
        tid_to_fs_dict = {}
        
        for tid, count in tid_message_counts.items():
            if count > 5000 and tid != 0:
                error = [abs(3600 - count/fs) for fs in possible_freqs]
                tid_to_fs_dict[tid] = possible_freqs[np.argmin(error)]

        # 4. Merge Coarse and Fine Clocks to Create Global Time Array
        t_clock_data = np.zeros(transmitter_id_bytes.shape)
        t_clock_data[transmitter_id_bytes == 0] = 1 
        corse_time_vector = np.cumsum(t_clock_data) * self.clock_tick_cycle
        fine_time_vector = t_stamps_256 * self.clock_division
        global_time_array = fine_time_vector + corse_time_vector

        results = []

        # 5. Process Each Transmitter for Signal Loss
        for tid, fs in tid_to_fs_dict.items():
            
            # Isolate time and timestamps for the specific transmitter
            tid_time_raw = global_time_array[transmitter_id_bytes == tid]
            tid_t_stamps_256 = t_stamps_256[transmitter_id_bytes == tid]
            raw_count = len(tid_time_raw)

            # --- A. Bad Message Identification ---
            n_messages = fs / 128
            expected_interval = 256 / n_messages 
            timestamp_moduli = tid_t_stamps_256 % expected_interval

            n_rows = int(fs * 4)
            n_fullcols = int(timestamp_moduli.size // n_rows)
            n_extra_stamps = timestamp_moduli.shape[0] - (n_rows * n_fullcols)

            # Drift correction logic
            if n_extra_stamps > 0:
                end_moduli = timestamp_moduli[-n_extra_stamps:]
                reshaped_moduli = np.reshape(timestamp_moduli[:-n_extra_stamps], (n_rows, n_fullcols), order='F')
                end_mean = ss.circmean(end_moduli, high=expected_interval)
                end_moduli_corrected = (end_moduli - end_mean)
                mean_vector = ss.circmean(reshaped_moduli, high=expected_interval, axis=0)
                moduli_array_corrected = (reshaped_moduli - mean_vector)
                drift_corrected = np.concatenate([np.ravel(moduli_array_corrected, order='F'), end_moduli_corrected])
            else:
                reshaped_moduli = np.reshape(timestamp_moduli, (n_rows, n_fullcols), order='F')
                mean_vector = ss.circmean(reshaped_moduli, high=expected_interval, axis=0)
                moduli_array_corrected = (reshaped_moduli - mean_vector)
                drift_corrected = np.ravel(moduli_array_corrected, order='F')

            drift_corrected = np.absolute(drift_corrected)
            
            # Identify bad messages
            bad_message_locs = np.where(np.logical_and(
                drift_corrected > 9,
                drift_corrected < (expected_interval - 9)
            ))[0]
            
            bad_message_count = len(bad_message_locs)

            # --- B. Gap Quantification ---
            # Remove bad messages before calculating physical time gaps
            cleaned_time_array = np.delete(tid_time_raw, bad_message_locs)
            
            if len(cleaned_time_array) > 1:
                diff_array = np.diff(cleaned_time_array)
                gaps = diff_array[diff_array > self.gap_threshold]
                num_gaps = len(gaps)
                total_gap_duration = np.sum(gaps) if num_gaps > 0 else 0.0
            else:
                num_gaps = 0
                total_gap_duration = 0.0

            # Compile
            results.append({
                'Transmitter_ID': tid,
                'Sampling_Rate_Hz': fs,
                'Raw_Messages': raw_count,
                'Bad_Messages_Dropped': bad_message_count,
                f'Gaps_Over_{self.gap_threshold}s': num_gaps,
                'Total_Missing_Seconds': round(total_gap_duration, 3)
            })

        return pd.DataFrame(results)


In [3]:

# ==========================================
# Application usage example:
# ==========================================
if __name__ == "__main__":
    file_to_test = "/home/jovyan/work/epimirna_expipeconversion/quantify_signal_dropout/M1491692352.ndf"
    
    # Initialize the quantifier
    analyzer = StandaloneSignalLossQuantifier(file_to_test, gap_threshold_seconds=1.0)
    
    # Generate the pandas DataFrame report
    loss_report = analyzer.quantify()
    print(loss_report.to_string(index=False))

 Transmitter_ID  Sampling_Rate_Hz  Raw_Messages  Bad_Messages_Dropped  Gaps_Over_1.0s  Total_Missing_Seconds
             19               512       1840964                  6396               0                  0.000
             10               512       1832696                  1325               0                  0.000
             13               512       1832670                   201               0                  0.000
             12               512       1832484                  5969               0                  0.000
              5               512       1830700                  1625               0                  0.000
              3               512       1827651                  4243               0                  0.000
              4               512       1825598                  4199               0                  0.000
              1               512       1813634                  6908               0                  0.000
            252    